<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW5/HW5_ViT_Transformer_Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchsummary import summary
!pip install torchinfo
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib.pyplot as plt

import time

torch.manual_seed(1)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#Problem 1

In [2]:
# Hyperparameters
image_size = 32
num_classes = 100
num_epochs = 10
batch_size = 64
learning_rate = 0.001

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_data = torchvision.datasets.CIFAR100(
    root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR100(
    root='./data', train=False, transform=transform, download=True)

In [5]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)

In [6]:
# Patch embedding layer
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                            kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # [B, embed_dim, H', W']
        x = x.flatten(2)  # [B, embed_dim, num_patches]
        x = x.transpose(1, 2)  # [B, num_patches, embed_dim]
        return x


# Transformer Encoder
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x2 = self.layer_norm1(x)
        attention_output, _ = self.attention(x2, x2, x2)
        x = x + attention_output
        x2 = self.layer_norm2(x)
        mlp_output = self.mlp(x2)
        x = x + mlp_output
        return x

# Vision Transformer
class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, embed_dim,
                 num_heads, num_layers, dropout=0.1):
        super().__init__()
        mlp_dim = embed_dim * 4
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        num_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.ModuleList(
            [TransformerEncoder(embed_dim, num_heads, mlp_dim, dropout)
             for _ in range(num_layers)]
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        for transformer in self.transformer:
            x = transformer(x)

        x = self.layer_norm(x)
        cls_token_final = x[:, 0]
        x = self.head(cls_token_final)
        return x

In [7]:
# Training loop
def train():
  model.train()
  for epoch in range(num_epochs):
      for i, (images, labels) in enumerate(train_loader):
          images = images.to(device)
          labels = labels.to(device)

          # Forward pass
          outputs = model(images)
          loss = criterion(outputs, labels)

          # Backward and optimize
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          if (i+1) % 100 == 0:
              print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}')

In [8]:
# Test the model
def test():
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy: {100 * correct / total}%')

In [9]:
# Run training and testing
def run():
  if __name__ == '__main__':
      print("Training started...")
      start_event = torch.cuda.Event(True)
      end_event = torch.cuda.Event(True)
      start_event.record()
      train()
      end_event.record()
      torch.cuda.synchronize()
      elapsed_time_ms = start_event.elapsed_time(end_event)
      print(f"Training execution time: {(elapsed_time_ms / 1000):.3f} s")
      print("\nTesting started...")
      test()
  from torchinfo import summary

  dummy_input = torch.randn(batch_size, 3, image_size, image_size).to(device)

  # Get model summary including FLOPs and parameters
  model_summary = summary(model, input_data=dummy_input, verbose=0)

  print(f"\nTotal theoretical parameters: {model_summary.total_params}")
  print(f"Total FLOPs per forward pass with batch size {batch_size}: {2 * model_summary.total_mult_adds}")

## Training and Testing

### ViT Patch Size 4, Embedding Dimension 256, Heads 4, Layers 4

In [ ]:
model = VisionTransformer(
    image_size=image_size,
    num_classes=num_classes,
    patch_size=4,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

run()

Training started...
Epoch [1/10], Step [100], Loss: 4.7552
Epoch [1/10], Step [200], Loss: 4.6071
Epoch [1/10], Step [300], Loss: 4.6391
Epoch [1/10], Step [400], Loss: 4.6334
Epoch [1/10], Step [500], Loss: 4.6456
Epoch [1/10], Step [600], Loss: 4.6308
Epoch [1/10], Step [700], Loss: 4.6564
Epoch [2/10], Step [100], Loss: 4.6535
Epoch [2/10], Step [200], Loss: 4.5936
Epoch [2/10], Step [300], Loss: 4.6722
Epoch [2/10], Step [400], Loss: 4.6107
Epoch [2/10], Step [500], Loss: 4.6258
Epoch [2/10], Step [600], Loss: 4.6525
Epoch [2/10], Step [700], Loss: 4.6170
Epoch [3/10], Step [100], Loss: 4.6240
Epoch [3/10], Step [200], Loss: 4.6128
Epoch [3/10], Step [300], Loss: 4.5970
Epoch [3/10], Step [400], Loss: 4.5992
Epoch [3/10], Step [500], Loss: 4.6346
Epoch [3/10], Step [600], Loss: 4.6336
Epoch [3/10], Step [700], Loss: 4.6099
Epoch [4/10], Step [100], Loss: 4.6414
Epoch [4/10], Step [200], Loss: 4.6113
Epoch [4/10], Step [300], Loss: 4.6116
Epoch [4/10], Step [400], Loss: 4.6119
Epoch

### ViT Embedding Patch Size 4, Embedding Dimension 512, Heads 4, Layers 8

In [ ]:
model = VisionTransformer(
    image_size=image_size,
    num_classes=num_classes,
    patch_size=4,
    embed_dim=512,
    num_heads=4,
    num_layers=8,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

run()

Training started...
Epoch [1/10], Step [100], Loss: 4.6271
Epoch [1/10], Step [200], Loss: 4.6161
Epoch [1/10], Step [300], Loss: 4.7209
Epoch [1/10], Step [400], Loss: 4.6080
Epoch [1/10], Step [500], Loss: 4.6131
Epoch [1/10], Step [600], Loss: 4.6715
Epoch [1/10], Step [700], Loss: 4.6131
Epoch [2/10], Step [100], Loss: 4.6440
Epoch [2/10], Step [200], Loss: 4.6068
Epoch [2/10], Step [300], Loss: 4.6189
Epoch [2/10], Step [400], Loss: 4.6203
Epoch [2/10], Step [500], Loss: 4.6421
Epoch [2/10], Step [600], Loss: 4.6367
Epoch [2/10], Step [700], Loss: 4.6835
Epoch [3/10], Step [100], Loss: 4.6037
Epoch [3/10], Step [200], Loss: 4.6272
Epoch [3/10], Step [300], Loss: 4.6286
Epoch [3/10], Step [400], Loss: 4.6068
Epoch [3/10], Step [500], Loss: 4.6539
Epoch [3/10], Step [600], Loss: 4.6198
Epoch [3/10], Step [700], Loss: 4.5873
Epoch [4/10], Step [100], Loss: 4.5924
Epoch [4/10], Step [200], Loss: 4.6199
Epoch [4/10], Step [300], Loss: 4.6150
Epoch [4/10], Step [400], Loss: 4.6314
Epoch

### ViT Embedding Patch Size 4, Embedding Dimension 512, Heads 8, Layers 8

In [ ]:
model = VisionTransformer(
    image_size=image_size,
    num_classes=num_classes,
    patch_size=4,
    embed_dim=512,
    num_heads=8,
    num_layers=8,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

run()

Training started...
Epoch [1/10], Step [100], Loss: 4.6566
Epoch [1/10], Step [200], Loss: 4.6318
Epoch [1/10], Step [300], Loss: 4.6912
Epoch [1/10], Step [400], Loss: 4.7076
Epoch [1/10], Step [500], Loss: 4.7079
Epoch [1/10], Step [600], Loss: 4.5711
Epoch [1/10], Step [700], Loss: 4.6717
Epoch [2/10], Step [100], Loss: 4.6872
Epoch [2/10], Step [200], Loss: 4.6425
Epoch [2/10], Step [300], Loss: 4.5535
Epoch [2/10], Step [400], Loss: 4.6797
Epoch [2/10], Step [500], Loss: 4.6191
Epoch [2/10], Step [600], Loss: 4.5950
Epoch [2/10], Step [700], Loss: 4.6061
Epoch [3/10], Step [100], Loss: 4.6265
Epoch [3/10], Step [200], Loss: 4.6185
Epoch [3/10], Step [300], Loss: 4.5611
Epoch [3/10], Step [400], Loss: 4.6249
Epoch [3/10], Step [500], Loss: 4.6168
Epoch [3/10], Step [600], Loss: 4.6045
Epoch [3/10], Step [700], Loss: 4.6375
Epoch [4/10], Step [100], Loss: 4.6286
Epoch [4/10], Step [200], Loss: 4.5868
Epoch [4/10], Step [300], Loss: 4.6221
Epoch [4/10], Step [400], Loss: 4.6042
Epoch

### ViT Embedding Patch Size 8, Embedding Dimension 512, Heads 8, Layers 8

In [10]:
model = VisionTransformer(
    image_size=image_size,
    num_classes=num_classes,
    patch_size=8,
    embed_dim=512,
    num_heads=8,
    num_layers=8,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

run()

Training started...
Epoch [1/10], Step [100], Loss: 4.6315
Epoch [1/10], Step [200], Loss: 4.7640
Epoch [1/10], Step [300], Loss: 4.6574
Epoch [1/10], Step [400], Loss: 4.6665
Epoch [1/10], Step [500], Loss: 4.6964
Epoch [1/10], Step [600], Loss: 4.6449
Epoch [1/10], Step [700], Loss: 4.6326
Epoch [2/10], Step [100], Loss: 4.5672
Epoch [2/10], Step [200], Loss: 4.5944
Epoch [2/10], Step [300], Loss: 4.6230
Epoch [2/10], Step [400], Loss: 4.6426
Epoch [2/10], Step [500], Loss: 4.6893
Epoch [2/10], Step [600], Loss: 4.6246
Epoch [2/10], Step [700], Loss: 4.6157
Epoch [3/10], Step [100], Loss: 4.6301
Epoch [3/10], Step [200], Loss: 4.6368
Epoch [3/10], Step [300], Loss: 4.6165
Epoch [3/10], Step [400], Loss: 4.6512
Epoch [3/10], Step [500], Loss: 4.6188
Epoch [3/10], Step [600], Loss: 4.6008
Epoch [3/10], Step [700], Loss: 4.6229
Epoch [4/10], Step [100], Loss: 4.6237
Epoch [4/10], Step [200], Loss: 4.6652
Epoch [4/10], Step [300], Loss: 4.6334
Epoch [4/10], Step [400], Loss: 4.6110
Epoch

## Implementing ResNet-18

In [ ]:
# Initialize ResNet-18 model
model = resnet18(weights=None)

# Modify the final fully connected layer for 100 classes
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

run()

Training started...
Epoch [1/10], Step [100], Loss: 3.8778
Epoch [1/10], Step [200], Loss: 3.9824
Epoch [1/10], Step [300], Loss: 3.3577
Epoch [1/10], Step [400], Loss: 3.5566
Epoch [1/10], Step [500], Loss: 3.3325
Epoch [1/10], Step [600], Loss: 3.3133
Epoch [1/10], Step [700], Loss: 3.3000
Epoch [2/10], Step [100], Loss: 3.2819
Epoch [2/10], Step [200], Loss: 2.9621
Epoch [2/10], Step [300], Loss: 3.1324
Epoch [2/10], Step [400], Loss: 2.8350
Epoch [2/10], Step [500], Loss: 2.8729
Epoch [2/10], Step [600], Loss: 2.5135
Epoch [2/10], Step [700], Loss: 2.4860
Epoch [3/10], Step [100], Loss: 2.2746
Epoch [3/10], Step [200], Loss: 2.2307
Epoch [3/10], Step [300], Loss: 2.3801
Epoch [3/10], Step [400], Loss: 2.7125
Epoch [3/10], Step [500], Loss: 2.2419
Epoch [3/10], Step [600], Loss: 2.6133
Epoch [3/10], Step [700], Loss: 2.1344
Epoch [4/10], Step [100], Loss: 1.9715
Epoch [4/10], Step [200], Loss: 1.9934
Epoch [4/10], Step [300], Loss: 1.9939
Epoch [4/10], Step [400], Loss: 2.1689
Epoch